# 🩺 Projet ML : Prédiction du diabète (Pipeline → Pickle → FastAPI → Flask)

**Objectif** : prédire si un patient est atteint de diabète à partir de mesures médicales (dataset *Pima Indians Diabetes*, 768 patients, cible `Outcome` : 1 = diabète, 0 = non).

**Étapes du projet**
1. Chargement et exploration des données
2. Prétraitement (valeurs aberrantes `0` = valeurs manquantes, imputation, normalisation) regroupé dans un **Pipeline scikit-learn**
3. Comparaison de modèles + optimisation (GridSearchCV)
4. Évaluation
5. **Sérialisation** du pipeline complet avec `pickle` (`.pkl`)
6. **API FastAPI** qui charge le `.pkl` et expose `/predict`
7. **Interface Flask** qui appelle l'API FastAPI

## 1. Imports

In [ ]:
import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, accuracy_score)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Chargement des données

In [ ]:
URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
try:
    df = pd.read_csv(URL)
except Exception as e:
    print("Téléchargement impossible, uploadez diabetes.csv manuellement :", e)
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])

print(df.shape)
df.head()

## 3. Exploration (EDA)

In [ ]:
display(df.info())
display(df.describe().T)
print("\nRépartition de la cible :")
print(df["Outcome"].value_counts(normalize=True).round(3))

In [ ]:
# Dans ce dataset, un 0 est IMPOSSIBLE biologiquement pour ces colonnes => valeurs manquantes déguisées
cols_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Nombre de zéros par colonne :")
print((df[cols_zero] == 0).sum())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), df.columns[:-1]):
    sns.histplot(data=df, x=col, hue="Outcome", kde=True, ax=ax)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de corrélation"); plt.show()

## 4. Séparation train / test
On sépare **avant** tout prétraitement pour éviter la fuite de données (*data leakage*) : l'imputation et la normalisation seront apprises uniquement sur le train grâce au Pipeline.

In [ ]:
X = df.drop(columns="Outcome")
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## 5. Pipeline de prétraitement
- Colonnes avec `0` aberrant → imputation par la **médiane** (`missing_values=0`)
- Toutes les colonnes → **StandardScaler**
- Puis le modèle

Le tout est dans un seul objet `Pipeline` : une seule sérialisation, et l'API n'a besoin d'aucun code de prétraitement.

In [ ]:
cols_other = [c for c in X.columns if c not in cols_zero]

preprocess = ColumnTransformer([
    ("zero_to_median", SimpleImputer(missing_values=0, strategy="median"), cols_zero),
    ("passthrough", "passthrough", cols_other),
])

def make_pipeline(model):
    return Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

## 6. Comparaison de plusieurs modèles (validation croisée)

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
res = {}
for name, m in models.items():
    scores = cross_val_score(make_pipeline(m), X_train, y_train, cv=cv, scoring="roc_auc")
    res[name] = scores.mean()
    print(f"{name:20s} AUC CV = {scores.mean():.3f} (+/- {scores.std():.3f})")

## 7. Optimisation du meilleur modèle (GridSearchCV)
La **régression logistique** obtient le meilleur AUC en validation croisée : on l'optimise (`C`, `class_weight` pour mieux détecter les malades).

In [ ]:
pipe = make_pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)
print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur AUC CV      :", round(grid.best_score_, 3))
best_pipeline = grid.best_estimator_

## 8. Évaluation sur le jeu de test

In [ ]:
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("AUC      :", round(roc_auc_score(y_test, y_proba), 3))
print(classification_report(y_test, y_pred, target_names=["Sain", "Diabète"]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["Sain", "Diabète"]).plot(ax=ax[0], cmap="Blues")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax[1])
plt.show()

## 9. Sérialisation du pipeline avec pickle (.pkl)

In [ ]:
import os
os.makedirs("model", exist_ok=True)
with open("model/diabetes_pipeline.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

# Test de rechargement + prédiction sur un nouveau patient
with open("model/diabetes_pipeline.pkl", "rb") as f:
    loaded = pickle.load(f)

patient = pd.DataFrame([{
    "Pregnancies": 6, "Glucose": 148, "BloodPressure": 72, "SkinThickness": 35,
    "Insulin": 0, "BMI": 33.6, "DiabetesPedigreeFunction": 0.627, "Age": 50}])
print("Prédiction :", loaded.predict(patient)[0], "| proba :", loaded.predict_proba(patient)[0][1].round(3))

## 10. API FastAPI
On écrit le fichier `api/main.py` : il charge le `.pkl` au démarrage et expose `POST /predict`.

In [ ]:
import os
os.makedirs('api', exist_ok=True)

In [ ]:
%%writefile api/main.py
"""API FastAPI : charge le pipeline sérialisé (pkl) et expose /predict."""
import os
import pickle
from pathlib import Path

import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

MODEL_PATH = Path(os.getenv("MODEL_PATH", Path(__file__).resolve().parent.parent / "model" / "diabetes_pipeline.pkl"))

with open(MODEL_PATH, "rb") as f:
    pipeline = pickle.load(f)

app = FastAPI(title="API Prédiction Diabète", version="1.0")


class Patient(BaseModel):
    Pregnancies: int = Field(..., ge=0, le=20, description="Nombre de grossesses")
    Glucose: float = Field(..., ge=0, le=300)
    BloodPressure: float = Field(..., ge=0, le=200)
    SkinThickness: float = Field(..., ge=0, le=100)
    Insulin: float = Field(..., ge=0, le=900)
    BMI: float = Field(..., ge=0, le=80)
    DiabetesPedigreeFunction: float = Field(..., ge=0, le=3)
    Age: int = Field(..., ge=1, le=120)


@app.get("/")
def root():
    return {"message": "API diabète OK", "docs": "/docs"}


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(patient: Patient):
    df = pd.DataFrame([patient.model_dump()])
    pred = int(pipeline.predict(df)[0])
    proba = float(pipeline.predict_proba(df)[0][1])
    return {
        "prediction": pred,
        "label": "Diabète détecté" if pred == 1 else "Pas de diabète",
        "probability": round(proba, 4),
    }

### Lancer l'API en arrière-plan dans Colab et la tester

In [ ]:
!pip -q install fastapi uvicorn
import subprocess, time, requests
api_proc = subprocess.Popen(["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)

r = requests.post("http://127.0.0.1:8000/predict", json={
    "Pregnancies": 6, "Glucose": 148, "BloodPressure": 72, "SkinThickness": 35,
    "Insulin": 0, "BMI": 33.6, "DiabetesPedigreeFunction": 0.627, "Age": 50})
print(r.status_code, r.json())

## 11. Interface Flask (appelle l'API FastAPI)

In [ ]:
import os
os.makedirs('web/templates', exist_ok=True)

In [ ]:
%%writefile web/app.py
"""Interface Flask : formulaire -> appel de l'API FastAPI -> affichage du résultat."""
import os

import requests
from flask import Flask, render_template, request

app = Flask(__name__)
API_URL = os.getenv("API_URL", "http://127.0.0.1:8000/predict")

FIELDS = [
    ("Pregnancies", "Grossesses", "int", "2"),
    ("Glucose", "Glucose (mg/dL)", "float", "120"),
    ("BloodPressure", "Pression artérielle (mm Hg)", "float", "70"),
    ("SkinThickness", "Épaisseur de peau (mm)", "float", "20"),
    ("Insulin", "Insuline (mu U/ml)", "float", "80"),
    ("BMI", "IMC", "float", "28.5"),
    ("DiabetesPedigreeFunction", "Fonction pedigree diabète", "float", "0.45"),
    ("Age", "Âge", "int", "35"),
]


@app.route("/", methods=["GET", "POST"])
def index():
    result, error, values = None, None, {}
    if request.method == "POST":
        try:
            payload = {}
            for name, _, typ, _ in FIELDS:
                values[name] = request.form[name]
                payload[name] = int(values[name]) if typ == "int" else float(values[name])
            r = requests.post(API_URL, json=payload, timeout=10)
            if r.status_code == 200:
                result = r.json()
            else:
                error = f"Erreur API ({r.status_code}) : {r.text}"
        except ValueError:
            error = "Valeurs invalides : vérifiez les champs."
        except requests.exceptions.ConnectionError:
            error = "Impossible de joindre l'API FastAPI. Est-elle lancée ?"
    return render_template("index.html", fields=FIELDS, result=result, error=error, values=values)


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)

In [ ]:
%%writefile web/templates/index.html
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Prédiction du diabète</title>
<style>
  body{font-family:system-ui,sans-serif;background:#f1f5f9;margin:0;padding:2rem;color:#0f172a}
  .card{max-width:560px;margin:auto;background:#fff;padding:2rem;border-radius:14px;box-shadow:0 4px 20px #0001}
  h1{margin-top:0;font-size:1.5rem}
  label{display:block;margin-top:.8rem;font-size:.9rem;font-weight:600}
  input{width:100%;padding:.55rem;margin-top:.25rem;border:1px solid #cbd5e1;border-radius:8px;box-sizing:border-box}
  button{margin-top:1.4rem;width:100%;padding:.8rem;background:#2563eb;color:#fff;border:0;border-radius:8px;font-size:1rem;cursor:pointer}
  button:hover{background:#1d4ed8}
  .res{margin-top:1.4rem;padding:1rem;border-radius:10px;text-align:center;font-weight:600}
  .pos{background:#fee2e2;color:#991b1b}.neg{background:#dcfce7;color:#166534}.err{background:#fef3c7;color:#92400e}
  small{display:block;margin-top:1rem;color:#64748b}
</style>
</head>
<body>
<div class="card">
  <h1>🩺 Prédiction du diabète</h1>
  <form method="post">
    {% for name, label, typ, default in fields %}
      <label>{{ label }}
        <input name="{{ name }}" type="number" step="{{ '1' if typ=='int' else 'any' }}" required
               value="{{ values.get(name, default) }}">
      </label>
    {% endfor %}
    <button type="submit">Prédire</button>
  </form>
  {% if result %}
    <div class="res {{ 'pos' if result.prediction==1 else 'neg' }}">
      {{ result.label }}<br>Probabilité : {{ (result.probability*100)|round(1) }} %
    </div>
  {% endif %}
  {% if error %}<div class="res err">{{ error }}</div>{% endif %}
  <small>Projet pédagogique : ne remplace pas un avis médical.</small>
</div>
</body>
</html>

### Lancer Flask dans Colab et l'afficher

In [ ]:
import threading, sys
sys.path.insert(0, "web")
import importlib, app as flask_module
importlib.reload(flask_module)

threading.Thread(target=lambda: flask_module.app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

from google.colab import output
output.serve_kernel_port_as_iframe(5000, height=800)

## 12. Télécharger tout le projet

In [ ]:
!zip -r projet_diabete.zip data model api web requirements.txt -x "*/__pycache__/*"
from google.colab import files
files.download("projet_diabete.zip")

## ✅ Conclusion
- Données prétraitées via un `Pipeline` (imputation + normalisation) → pas de fuite de données
- Modèle optimisé et sérialisé en `.pkl` (pipeline **complet**)
- **FastAPI** sert le modèle ; **Flask** fournit l'interface et appelle l'API via HTTP

**Pistes d'amélioration** : SMOTE / `class_weight` pour le déséquilibre, SHAP pour l'interprétation, Docker, tests unitaires.